# B2-022 — Practice p18

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** autoencoder

## Task

Train a tiny linear autoencoder on the seeded `lowrank8d` data and certify that it recovers the PCA subspace.

**Data.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `latent_data.train_tensor("lowrank8d")` of shape `(48,8)` and `latent_data.heldout_tensor("lowrank8d")` of shape `(12,8)`, built from the immutable `latent_data.TRAIN_IDS["lowrank8d"]` and `latent_data.HELDOUT_IDS["lowrank8d"]` in stored order. Pass training rows to the model exactly as returned (same float32 values; no normalization, noise, shuffling, or concatenation), so every row the model's `forward` receives during training hashes, with `latent_data.row_sha256`, into the training part of `latent_data.ROW_SHA256["lowrank8d"]`. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice.

**Model.** `class LinearAutoencoder(nn.Module)` creates, in this order, `self.encoder = nn.Linear(8, 2)` and `self.decoder = nn.Linear(2, 8)` (biases on), with `forward(x)` returning `self.decoder(self.encoder(x))`. Call `torch.manual_seed(SEED)` immediately before `model = LinearAutoencoder()`.

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=0.02, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_linear_autoencoder(model, batch, optimizer)` that performs **exactly 1500** updates on `batch` with `F.mse_loss(model(batch), batch)` (default mean reduction) and returns a list of 1500 Python floats. Each update runs, in this order: `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()`. Full batch, no shuffling, no scheduler, no gradient clipping, no accumulation, no early stopping. Call it as `trace = train_linear_autoencoder(model, train_rows(), optimizer)`.

**Subspace certificate.** Write `projector(W)` returning `W @ torch.linalg.solve(W.T @ W, W.T)` (no explicit inverse) for a float64 `(8,2)` matrix. The PCA reference uses training rows only: `centered = T - T.mean(dim=0)` for `T = train_rows().double()`, `_, _, Vh = torch.linalg.svd(centered, full_matrices=False)`, `W_pca = Vh[:2].T`. The autoencoder subspace uses `W_ae = model.decoder.weight.detach().double()` (shape `(8,2)`). Define `distance = torch.linalg.norm(projector(W_ae) - projector(W_pca)).item()`.

**Parts.**
(A) Record `distance_before` on the freshly constructed model and the held-out MSE before training.
(B) Train; record `distance_after` and the held-out MSE after training.
(C) Certify: `len(trace) == 1500`; `distance_before >= 0.5`; `distance_after <= 0.02`; held-out MSE after training `<= 0.03`; `projector(W_ae)` is symmetric and idempotent (`atol = rtol = 1e-8`); the decoder weight changed; split hashes as in p17.
The `0.02` tolerance was set from the frozen-seed reference run (whose distance is below `0.002`) with a ten-fold margin; it is not an a priori guarantee.
(D) Explain why `torch.allclose(W_ae, W_pca, ...)` would be the wrong certificate even for a perfectly trained model, and why the PCA reference must be centered.

**Required answer-check assertions.** Every item of (C), plus `projector(W_pca)` having trace `2` within `1e-8`.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.